# Train parasite-egg detector (Chula-ParasiteEgg-11) on Kaggle → ONNX

**Notebook settings (right sidebar):**
- Accelerator: **GPU T4 x2** (not P100: current PyTorch builds may not support it)
- Internet: **On** (needs a phone-verified Kaggle account)
- Add-ons → Secrets: add **GH_TOKEN** (GitHub token, the repo is private) and attach it to this notebook

**How to run:** *Save Version → Save & Run All (Commit)*. It runs in the background for up to 12 h (browser can be closed) and everything in `/kaggle/working` (checkpoints, ONNX) is saved as the version's output.

**Didn't finish in one run?** Add this notebook's previous version output as an Input (*Add Input → Your Work → Notebooks*), then Save & Run All again. The resume cell finds `last.pt` under `/kaggle/input` and continues. A Colab checkpoint folder (`runs/detect/parasite` from Drive) uploaded as a Kaggle dataset works the same way.

**Faster data setup (optional):** upload `yolo.tar` from Drive (`MyDrive/parasite_data/`) as a Kaggle dataset and add it as an Input; otherwise the dataset is downloaded from Zenodo and converted each run.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
if torch.cuda.is_available() and torch.cuda.get_device_capability(0) < (7, 0):
    print("WARNING: old GPU (P100?). Switch the accelerator to GPU T4 x2.")

In [ ]:
# Repo is cloned to /tmp so the token in .git/config never ends up in the saved output.
# Only runs/ (symlinked into /kaggle/working) and the exported model are saved.
from kaggle_secrets import UserSecretsClient
try:
    tok = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    tok = None
url = f"https://{tok}@github.com/rtpsticky/ParasiteEgg.git" if tok else "https://github.com/rtpsticky/ParasiteEgg.git"
%cd /tmp
!rm -rf ParasiteEgg && git clone -q {url}
%cd /tmp/ParasiteEgg
!mkdir -p /kaggle/working/runs && ln -sfn /kaggle/working/runs runs
!pip -q install -r requirements.txt

## Dataset → /tmp/yolo
Uses a prepared dataset from /kaggle/input if one is attached (folder with data.yaml, or yolo.tar), else downloads from Zenodo (record 15866298, MIT license) and converts.

In [ ]:
%%bash
set -e
OUT=/tmp/yolo
YAML=$(find /kaggle/input -name data.yaml -path '*yolo*' 2>/dev/null | head -n1 || true)
TAR=$(find /kaggle/input -name 'yolo.tar' 2>/dev/null | head -n1 || true)
if [ -d "$OUT/images" ]; then
  echo "dataset already at $OUT"
elif [ -n "$YAML" ]; then
  echo "copying prepared dataset $(dirname "$YAML")"
  cp -r "$(dirname "$YAML")" "$OUT"
elif [ -n "$TAR" ]; then
  echo "extracting $TAR"
  tar -xf "$TAR" -C /tmp
else
  mkdir -p /tmp/raw && cd /tmp/raw
  B=https://zenodo.org/records/15866298/files
  wget -q -c "$B/Chula-ParasiteEgg-11.zip?download=1" -O train.zip
  wget -q -c "$B/Chula-ParasiteEgg-11_test.zip?download=1" -O test.zip
  wget -q -c "$B/test_labels_200.json?download=1" -O test_labels_200.json
  echo "1621521969547b0291c0cfbf7e450074  train.zip" | md5sum -c
  echo "a47fdab77ced45bc40b0e8b1e6ff8d22  test.zip" | md5sum -c
  unzip -q train.zip -d train && rm train.zip
  unzip -q test.zip -d test && rm test.zip
  cd /tmp/ParasiteEgg
  python scripts/prepare_dataset.py --src /tmp/raw/train --out $OUT \
    --test-src /tmp/raw/test --test-json /tmp/raw/test_labels_200.json --workers 4
  rm -rf /tmp/raw
fi
# point data.yaml at this location; alias the Colab path so Colab checkpoints resume too
sed -i "s|^path:.*|path: $OUT|" $OUT/data.yaml
mkdir -p /content && ln -sfn $OUT /content/yolo
cat $OUT/data.yaml
du -sh $OUT

## Train (or resume)
Kaggle gives 4 CPU cores, so `WORKERS = 4`. For both T4s set `DEVICE = "0,1"` and `BATCH = 32` (auto-batch only works on one GPU).

In [ ]:
import glob, os, shutil

MODEL, EPOCHS = "yolo11n.pt", 100
DEVICE, BATCH, WORKERS = "0", -1, 4

run_dir = "/kaggle/working/runs/detect/parasite"
if not os.path.exists(f"{run_dir}/weights/last.pt"):
    # also matches older outputs saved under .../runs/detect/runs/detect/parasite
    prev = sorted(glob.glob("/kaggle/input/**/parasite/weights/last.pt", recursive=True))
    if prev:
        src = os.path.dirname(os.path.dirname(prev[0]))
        print("resuming from", src)
        shutil.copytree(src, run_dir)

if os.path.exists(f"{run_dir}/weights/last.pt"):
    # if that run already finished this just prints an error; export below still runs
    !python scripts/train.py --resume --data /tmp/yolo/data.yaml --device {DEVICE} --batch {BATCH} --workers {WORKERS}
else:
    !python scripts/train.py --data /tmp/yolo/data.yaml --model {MODEL} --epochs {EPOCHS} --device {DEVICE} --batch {BATCH} --workers {WORKERS}

In [ ]:
import pandas as pd
csv = f"{run_dir}/results.csv"
if os.path.exists(csv):
    r = pd.read_csv(csv); r.columns = r.columns.str.strip()
    print(r[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(15).to_string(index=False))
else:  # don't fail the commit run here, or the export cell below never runs
    print("no results.csv at", csv, "| found:", glob.glob("/kaggle/working/**/results.csv", recursive=True))

## Evaluate on test + export ONNX → /kaggle/working/web_model

In [ ]:
!python scripts/export_onnx.py --weights {run_dir}/weights/best.pt --data /tmp/yolo/data.yaml --out /kaggle/working/web_model
!ls -la /kaggle/working/web_model